In [1]:
import sys
print("python exe:", sys.executable)   # full path to the interpreter
print("python ver:", sys.version)

# try torch only in the working notebook
try:
    import torch
    print("torch      :", torch.__version__, torch.__file__)
except ModuleNotFoundError as e:
    print("torch not importable:", e)



python exe: c:\Users\aneek\anaconda3\envs\tf_gpu_env\python.exe
python ver: 3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:49:16) [MSC v.1929 64 bit (AMD64)]


c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch      : 1.12.1+cu113 c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\torch\__init__.py


In [2]:
import tensorflow as tf
print(tf.__version__)  # This should print the version of TensorFlow
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

print("CUDA version:", tf.sysconfig.get_build_info()["cuda_version"])
print("cuDNN version:", tf.sysconfig.get_build_info()["cudnn_version"])

from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

2.10.0
Num GPUs Available:  1
CUDA version: 64_112
cuDNN version: 64_8
[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 3651367723157444346
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 5713690624
locality {
  bus_id: 1
  links {
  }
}
incarnation: 4243232330984310124
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9"
xla_global_id: 416903419
]


In [3]:
import time
import tensorflow as tf
import psutil

class PowerMonitor:
    def __init__(self):
        self.gpu_available = tf.config.list_physical_devices('GPU')
        
        # Hardware power specifications (adjust these values for your system)
        self.cpu_tdp = 65    # Typical TDP for desktop CPUs in watts
        self.gpu_tdp = 250   # Typical TDP for desktop GPUs in watts
        
    def get_stats(self):
        """Get system stats with power estimation"""
        stats = {
            'timestamp': time.time(),
            'cpu_%': psutil.cpu_percent(interval=0.1),
            'ram_mb': psutil.virtual_memory().used / (1024**2),
            'gpu_mem_mb': 0,
            'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85  # Base CPU power
        }
        
        if self.gpu_available:
            try:
                # TensorFlow GPU memory monitoring
                mem_info = tf.config.experimental.get_memory_info('GPU:0')
                stats.update({
                    'gpu_mem_mb': mem_info['current'] / (1024**2),
                    'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85 + 
                              self.gpu_tdp * 0.5 * 0.75  # Add GPU power estimate
                })
            except:
                pass
                
        return stats

# Initialize monitor
monitor = PowerMonitor()

In [4]:
import time
import torch
import psutil
import os

class PowerMonitor1:
    def __init__(self):
        self.gpu_available = torch.cuda.is_available()
        self.process = psutil.Process(os.getpid())  # Track current process
        
        # Hardware power specifications
        self.cpu_tdp = 65
        self.gpu_tdp = 250
        
    def get_stats(self):
        """Get process-specific stats with power estimation"""
        process_memory = self.process.memory_info()
        
        stats = {
            'timestamp': time.time(),
            'cpu_%': psutil.cpu_percent(interval=0.1),
            'process_ram_mb': process_memory.rss / (1024**2),  # Only this process's RAM
            'gpu_mem_mb': 0,
            'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85
        }
        
        if self.gpu_available:
            try:
                gpu_memory_allocated = torch.cuda.memory_allocated()
                stats.update({
                    'gpu_mem_mb': gpu_memory_allocated / (1024**2),
                    'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85 + 
                              self.gpu_tdp * 0.5 * 0.75
                })
            except Exception as e:
                print(f"Error retrieving GPU memory: {e}")
                
        return stats

# Initialize monitor1
monitor1 = PowerMonitor1()

# Model

In [5]:
# ============================================================
# DINOV2-BASE: IMPORTS AND CONFIGURATION
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from PIL import Image

from transformers import (
    AutoImageProcessor,
    AutoModel
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    auc
)


MODEL_NAME = "facebook/dinov2-base"

INPUT_SIZE = 160
NUM_CLASSES = 2
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4
RANDOM_SEED = 42


device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )

torch.backends.cudnn.enabled = True
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
# ============================================================
# LOAD DINOV2 PROCESSOR
# ============================================================

processor = AutoImageProcessor.from_pretrained(
    MODEL_NAME,
    size={
        "height": INPUT_SIZE,
        "width": INPUT_SIZE
    },
    do_center_crop=False
)


# Explicitly enforce the common benchmark resolution
processor.size = {
    "height": INPUT_SIZE,
    "width": INPUT_SIZE
}

if hasattr(processor, "crop_size"):
    processor.crop_size = {
        "height": INPUT_SIZE,
        "width": INPUT_SIZE
    }

if hasattr(processor, "do_center_crop"):
    processor.do_center_crop = False


# ============================================================
# DINOV2 CLASSIFIER
# ============================================================

class DinoV2Classifier(nn.Module):
    """
    Pretrained DINOv2-Base backbone with a new binary
    classification head.

    Labels:
        0 = real
        1 = fake
    """

    def __init__(
        self,
        model_name,
        num_classes=2,
        dropout_rate=0.3
    ):
        super().__init__()

        self.dinov2 = AutoModel.from_pretrained(
            model_name
        )

        hidden_size = (
            self.dinov2.config.hidden_size
        )

        self.dropout = nn.Dropout(
            dropout_rate
        )

        self.classifier = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self, pixel_values):

        outputs = self.dinov2(
            pixel_values=pixel_values
        )

        # CLS-token representation
        cls_features = (
            outputs.last_hidden_state[:, 0, :]
        )

        cls_features = self.dropout(
            cls_features
        )

        logits = self.classifier(
            cls_features
        )

        return logits


model = DinoV2Classifier(
    model_name=MODEL_NAME,
    num_classes=NUM_CLASSES,
    dropout_rate=0.3
)


# Full-model fine-tuning
for parameter in model.parameters():
    parameter.requires_grad = True


model = model.to(device)


trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)


print("Model:", MODEL_NAME)
print("Output classes:", NUM_CLASSES)
print("Trainable parameters:", trainable_parameters)
print("Total parameters:", total_parameters)
# ============================================================
# DINOV2 DATASET
# ============================================================

class DeepfakeDinoV2Dataset(Dataset):
    """
    Dataset for OpenCV-loaded images.

    Input images:
        OpenCV BGR NumPy arrays, PIL images,
        or file paths.

    Labels:
        0 = real
        1 = fake
    """

    def __init__(
        self,
        images,
        labels,
        processor
    ):
        self.images = images

        self.labels = np.asarray(
            labels,
            dtype=np.int64
        )

        self.processor = processor

        if len(self.images) != len(self.labels):
            raise ValueError(
                "The numbers of images and labels "
                "do not match."
            )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):

        image = self.images[index]
        label = int(self.labels[index])

        # OpenCV NumPy array
        if isinstance(image, np.ndarray):

            if (
                image.ndim != 3
                or image.shape[-1] != 3
            ):
                raise ValueError(
                    f"Invalid image shape at index "
                    f"{index}: {image.shape}"
                )

            if image.dtype != np.uint8:

                if image.max() <= 1.0:
                    image = (
                        image * 255.0
                    ).clip(
                        0,
                        255
                    ).astype(np.uint8)

                else:
                    image = image.clip(
                        0,
                        255
                    ).astype(np.uint8)

            # OpenCV BGR -> RGB
            image = image[..., ::-1]

            image = np.ascontiguousarray(
                image
            )

            image = Image.fromarray(
                image
            )

        # Image path
        elif isinstance(image, str):

            with Image.open(image) as opened_image:
                image = opened_image.convert(
                    "RGB"
                )

        # PIL image
        elif isinstance(image, Image.Image):

            image = image.convert("RGB")

        else:
            raise TypeError(
                f"Unsupported image type at index "
                f"{index}: {type(image)}"
            )

        processed = self.processor(
            images=image,
            return_tensors="pt"
        )

        pixel_values = processed[
            "pixel_values"
        ].squeeze(0)

        return (
            pixel_values,
            torch.tensor(
                label,
                dtype=torch.long
            )
        )

Device: cuda


c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model: facebook/dinov2-base
Output classes: 2
Trainable parameters: 86582018
Total parameters: 86582018


In [6]:
# ============================================================
# LOSS AND OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)
# ============================================================
# DINOV2 VALIDATION AND COMPLETE TEST EVALUATION
# ============================================================

def evaluate_dinov2(
    model,
    data_loader,
    criterion,
    device,
    return_details=False
):
    """
    return_details=False:
        Return validation loss and validation accuracy.

    return_details=True:
        Return complete binary test metrics.

    Labels:
        0 = real
        1 = fake
    """

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_labels = []
    all_predictions = []
    all_fake_probabilities = []

    with torch.inference_mode():

        for images, labels in data_loader:

            images = images.to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            labels = labels.to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )

            predictions = torch.argmax(
                logits,
                dim=1
            )

            fake_probabilities = probabilities[
                :,
                1
            ]

            batch_size = labels.size(0)

            total_loss += (
                loss.item() * batch_size
            )

            total_samples += batch_size

            all_labels.extend(
                labels.detach().cpu().numpy()
            )

            all_predictions.extend(
                predictions.detach().cpu().numpy()
            )

            all_fake_probabilities.extend(
                fake_probabilities
                .detach()
                .cpu()
                .numpy()
            )

    if total_samples == 0:
        raise RuntimeError(
            "The supplied DataLoader contains no samples."
        )

    average_loss = (
        total_loss / total_samples
    )

    y_true = np.asarray(
        all_labels,
        dtype=np.int64
    )

    y_pred = np.asarray(
        all_predictions,
        dtype=np.int64
    )

    y_score = np.asarray(
        all_fake_probabilities,
        dtype=np.float64
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    # Used during epoch-level validation
    if not return_details:
        return average_loss, accuracy

    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = confusion.ravel()

    # ========================================================
    # THRESHOLD-DEPENDENT METRICS
    # ========================================================

    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred
        )
    )

    precision = precision_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    f1 = f1_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_true,
        y_pred
    )

    false_positive_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else np.nan
    )

    false_negative_rate = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else np.nan
    )

    # ========================================================
    # PROBABILITY-BASED METRICS
    # ========================================================

    if len(np.unique(y_true)) == 2:

        roc_auc = roc_auc_score(
            y_true,
            y_score
        )

        average_precision = (
            average_precision_score(
                y_true,
                y_score
            )
        )

        pr_precision, pr_recall, _ = (
            precision_recall_curve(
                y_true,
                y_score,
                pos_label=1
            )
        )

        pr_auc = auc(
            pr_recall,
            pr_precision
        )

        roc_fpr, roc_tpr, roc_thresholds = (
            roc_curve(
                y_true,
                y_score,
                pos_label=1
            )
        )

        roc_fnr = 1.0 - roc_tpr

        eer_index = np.nanargmin(
            np.abs(
                roc_fpr - roc_fnr
            )
        )

        eer = (
            roc_fpr[eer_index]
            + roc_fnr[eer_index]
        ) / 2.0

        eer_threshold = (
            roc_thresholds[eer_index]
        )

    else:

        roc_auc = np.nan
        average_precision = np.nan
        pr_auc = np.nan
        eer = np.nan
        eer_threshold = np.nan

    # ========================================================
    # CLASSIFICATION REPORT
    # ========================================================

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=[
            "real",
            "fake"
        ],
        digits=4,
        zero_division=0
    )

    # ========================================================
    # RESULTS
    # ========================================================

    results = {
        "test_loss": average_loss,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall_sensitivity": recall,
        "specificity": specificity,
        "f1_score": f1,
        "mcc": mcc,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "average_precision": average_precision,
        "eer": eer,
        "eer_threshold": eer_threshold,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
        "number_of_test_images": int(
            total_samples
        )
    }

    predictions_df = pd.DataFrame({
        "true_label": y_true,
        "predicted_label": y_pred,
        "fake_probability": y_score
    })

    return (
        results,
        confusion,
        report,
        predictions_df
    )
# ============================================================
# DINOV2 TRAINING FUNCTION
# ============================================================

def train_dinov2(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    epochs=10
):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(epochs):

        model.train()

        total_train_loss = 0.0
        correct_train_predictions = 0
        total_train_samples = 0

        for images, labels in train_loader:

            images = images.to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            labels = labels.to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            optimizer.zero_grad()

            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

            loss.backward()
            optimizer.step()

            predictions = torch.argmax(
                logits,
                dim=1
            )

            batch_size = labels.size(0)

            total_train_loss += (
                loss.item() * batch_size
            )

            correct_train_predictions += (
                predictions == labels
            ).sum().item()

            total_train_samples += batch_size

        if total_train_samples == 0:
            raise RuntimeError(
                "The training DataLoader is empty."
            )

        train_loss = (
            total_train_loss
            / total_train_samples
        )

        train_accuracy = (
            correct_train_predictions
            / total_train_samples
        )

        val_loss, val_accuracy = (
            evaluate_dinov2(
                model=model,
                data_loader=val_loader,
                criterion=criterion,
                device=device,
                return_details=False
            )
        )

        history["train_loss"].append(
            train_loss
        )

        history["train_accuracy"].append(
            train_accuracy
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_accuracy"].append(
            val_accuracy
        )

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Accuracy: {train_accuracy:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Accuracy: {val_accuracy:.4f}"
        )

    return history

# Wild deepfake

In [7]:
import h5py
import numpy as np

H5_PATH = (
    r"D:\thesis\dataset\WildDeepfake\leakage_free_subset"
    r"\wilddeepfake_sequence_disjoint_face_preprocessed.h5"
)

with h5py.File(H5_PATH, "r") as h5f:
    # Load image arrays
    train_images = h5f["train_images"][:]
    train_labels = h5f["train_labels"][:]

    val_images = h5f["val_images"][:]
    val_labels = h5f["val_labels"][:]

    test_images = h5f["test_images"][:]
    test_labels = h5f["test_labels"][:]

# Verify dataset sizes
print(f"Total train: {len(train_images)} images")
print(f"Total validation: {len(val_images)} images")
print(f"Total test: {len(test_images)} images")

print(f"Train labels: {len(train_labels)}")
print(f"Validation labels: {len(val_labels)}")
print(f"Test labels: {len(test_labels)}")

# Verify shapes and data types
print("\nArray information:")
print(f"Train images: {train_images.shape}, dtype={train_images.dtype}")
print(f"Validation images: {val_images.shape}, dtype={val_images.dtype}")
print(f"Test images: {test_images.shape}, dtype={test_images.dtype}")

print(f"Train labels: {train_labels.shape}, dtype={train_labels.dtype}")
print(f"Validation labels: {val_labels.shape}, dtype={val_labels.dtype}")
print(f"Test labels: {test_labels.shape}, dtype={test_labels.dtype}")

# Verify class distributions
print("\nClass distribution:")
print(
    f"Train: Real={np.sum(train_labels == 0)}, "
    f"Fake={np.sum(train_labels == 1)}"
)
print(
    f"Validation: Real={np.sum(val_labels == 0)}, "
    f"Fake={np.sum(val_labels == 1)}"
)
print(
    f"Test: Real={np.sum(test_labels == 0)}, "
    f"Fake={np.sum(test_labels == 1)}"
)

Total train: 36000 images
Total validation: 6000 images
Total test: 18000 images
Train labels: 36000
Validation labels: 6000
Test labels: 18000

Array information:
Train images: (36000, 160, 160, 3), dtype=uint8
Validation images: (6000, 160, 160, 3), dtype=uint8
Test images: (18000, 160, 160, 3), dtype=uint8
Train labels: (36000,), dtype=uint8
Validation labels: (6000,), dtype=uint8
Test labels: (18000,), dtype=uint8

Class distribution:
Train: Real=9000, Fake=27000
Validation: Real=1500, Fake=4500
Test: Real=4500, Fake=13500


In [8]:
# ============================================================
# DATASETS
# ============================================================
train_dataset = DeepfakeDinoV2Dataset(images=train_images,labels=train_labels,processor=processor)
val_dataset = DeepfakeDinoV2Dataset(images=val_images,labels=val_labels,processor=processor)
test_dataset = DeepfakeDinoV2Dataset(images=test_images,labels=test_labels,processor=processor)
# ============================================================
# DATALOADERS
# ============================================================
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())


print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training samples: 36000
Validation samples: 6000
Testing samples: 18000
Training batches: 2250
Validation batches: 375
Testing batches: 1125


In [ ]:
history = train_dinov2(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=EPOCHS
)

Epoch 01/10 | Train Loss: 0.5743 | Train Accuracy: 0.7452 | Val Loss: 0.5261 | Val Accuracy: 0.7500
Epoch 02/10 | Train Loss: 0.5164 | Train Accuracy: 0.7502 | Val Loss: 0.5099 | Val Accuracy: 0.7473
Epoch 03/10 | Train Loss: 0.4745 | Train Accuracy: 0.7663 | Val Loss: 0.4733 | Val Accuracy: 0.7607
Epoch 04/10 | Train Loss: 0.4877 | Train Accuracy: 0.7622 | Val Loss: 0.5420 | Val Accuracy: 0.7577
Epoch 05/10 | Train Loss: 0.4944 | Train Accuracy: 0.7614 | Val Loss: 0.4927 | Val Accuracy: 0.7552
Epoch 06/10 | Train Loss: 0.4753 | Train Accuracy: 0.7799 | Val Loss: 0.5036 | Val Accuracy: 0.7510
Epoch 07/10 | Train Loss: 0.4508 | Train Accuracy: 0.7802 | Val Loss: 0.4841 | Val Accuracy: 0.7512
Epoch 08/10 | Train Loss: 0.4353 | Train Accuracy: 0.7913 | Val Loss: 0.4489 | Val Accuracy: 0.7742
Epoch 09/10 | Train Loss: 0.4753 | Train Accuracy: 0.8099 | Val Loss: 0.4736 | Val Accuracy: 0.7910
Epoch 10/10 | Train Loss: 0.4577 | Train Accuracy: 0.8122 | Val Loss: 0.4420 | Val Accuracy: 0.7977


In [10]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [ ]:
test_results, confusion, report, predictions_df = (
    evaluate_dinov2(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")
print(confusion)

print("\nConfusion Matrix Format:")
print("[[TN, FP],")
print(" [FN, TP]]")

print("\nClassification Report:")
print(report)
test_results_df = pd.DataFrame(
    [test_results]
)
display(test_results_df)

DINOV2-BASE TEST RESULTS
test_loss                     : 0.931596

Confusion Matrix:
accuracy                      : 0.791167

Confusion Matrix:
balanced_accuracy             : 0.665000

Confusion Matrix:
precision                     : 0.824117

Confusion Matrix:
recall_sensitivity            : 0.917333

Confusion Matrix:
specificity                   : 0.412667

Confusion Matrix:
f1_score                      : 0.868230

Confusion Matrix:
mcc                           : 0.384816

Confusion Matrix:
roc_auc                       : 0.813922

Confusion Matrix:
pr_auc                        : 0.928685

Confusion Matrix:
average_precision             : 0.928642

Confusion Matrix:
eer                           : 0.271407

Confusion Matrix:
eer_threshold                 : 0.997029

Confusion Matrix:
false_positive_rate           : 0.587333

Confusion Matrix:
false_negative_rate           : 0.082667

Confusion Matrix:
true_negatives                : 1857

Confusion Matrix:
false_positives    

,test_loss,accuracy,balanced_accuracy,precision,recall_sensitivity,specificity,f1_score,mcc,roc_auc,pr_auc,average_precision,eer,eer_threshold,false_positive_rate,false_negative_rate,true_negatives,false_positives,false_negatives,true_positives,number_of_test_images
0,0.931596,0.791167,0.665,0.824117,0.917333,0.412667,0.86823,0.384816,0.813922,0.928685,0.928642,0.271407,0.997029,0.587333,0.082667,1857,2643,1116,12384,18000


In [13]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
#print(f"RAM Used: {end['ram_mb'] - start['ram_mb']:.1f} MB")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 20.0%
Time Usage: 254.1 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [14]:
end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 20.6%
Time Usage: 254.2 s
GPU Memory Used: 1349.5 MB
Power Consumption: 93W


save the model

In [15]:
# ============================================================
# SAVE DINOV2-BASE CLASSIFIER
# ============================================================

import os
import torch


SAVE_DIR = os.path.join(
    r"D:\thesis\results",
    "dinov2_base_wild_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


WEIGHTS_PATH = os.path.join(
    SAVE_DIR,
    "dinov2_classifier_weights.pth"
)

CHECKPOINT_PATH = os.path.join(
    SAVE_DIR,
    "training_checkpoint.pt"
)


# ============================================================
# SAVE PROCESSOR
# ============================================================

processor.save_pretrained(
    SAVE_DIR
)


# ============================================================
# SAVE MODEL WEIGHTS ONLY
# ============================================================

torch.save(
    model.state_dict(),
    WEIGHTS_PATH
)


# ============================================================
# SAVE COMPLETE TRAINING CHECKPOINT
# ============================================================

completed_epochs = len(
    history.get(
        "train_loss",
        []
    )
)


checkpoint = {
    # Complete DINOv2 backbone and classifier-head weights
    "model_state_dict":
        model.state_dict(),

    # Optimizer state for resuming training
    "optimizer_state_dict":
        optimizer.state_dict(),

    # Training and validation history
    "history":
        history,

    # Model configuration
    "model_name":
        MODEL_NAME,

    "input_size":
        INPUT_SIZE,

    "num_classes":
        NUM_CLASSES,

    "dropout_rate":
        0.3,

    # Training configuration
    "completed_epochs":
        completed_epochs,

    "configured_epochs":
        EPOCHS,

    "batch_size":
        BATCH_SIZE,

    "learning_rate":
        LEARNING_RATE,

    "random_seed":
        RANDOM_SEED,

    # Label interpretation
    "label_mapping": {
        0: "real",
        1: "fake"
    },

    # Processor preprocessing information
    "image_mean":
        processor.image_mean,

    "image_std":
        processor.image_std,

    "processor_size":
        processor.size
}


torch.save(
    checkpoint,
    CHECKPOINT_PATH
)


# ============================================================
# VERIFY SAVED FILES
# ============================================================

print("DINOv2 model saved successfully.")
print("Save directory:", SAVE_DIR)
print("Weights:", WEIGHTS_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

print("\nSaved files:")

for filename in os.listdir(SAVE_DIR):

    file_path = os.path.join(
        SAVE_DIR,
        filename
    )

    if os.path.isfile(file_path):

        file_size_mb = (
            os.path.getsize(file_path)
            / (1024 ** 2)
        )

        print(
            f"{filename:40s}: "
            f"{file_size_mb:.2f} MB"
        )

DINOv2 model saved successfully.
Save directory: D:\thesis\results\dinov2_base_wild_160
Weights: D:\thesis\results\dinov2_base_wild_160\dinov2_classifier_weights.pth
Checkpoint: D:\thesis\results\dinov2_base_wild_160\training_checkpoint.pt

Saved files:
dinov2_classifier_weights.pth           : 330.37 MB
preprocessor_config.json                : 0.00 MB
training_checkpoint.pt                  : 991.11 MB


In [16]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_42624\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [17]:
# ============================================================
# PYTORCH DinoV2 WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        # MaxViT dataset returns:
        # images tensor, labels tensor
        images, labels = batch

        images = images.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # timm MaxViT returns logits directly
        logits = model(images)

        # Materialize part of the output
        _ = logits[-1, 0].item()


if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting DinoV2 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous CUDA operations remain queued
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for images, labels in test_loader:

            images = images.to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            # timm MaxViT forward pass
            logits = model(images)

            last_logits = logits

            processed_images += images.size(0)


    # Wait for all CUDA inference operations
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    if last_logits is None:
        monitor.stop(
            elapsed_seconds=0.0,
            number_of_images=0
        )

        raise RuntimeError(
            "No images were processed during inference."
        )


    last_output_value = float(
        last_logits[-1, 0]
        .detach()
        .cpu()
        .item()
    )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number

    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits
    del images


# ============================================================
# DISPLAY INDIVIDUAL RUNS
# ============================================================

results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual DinoV2 profiling runs:"
)

display(results_df)

Device: cuda
Test images: 18000
Test batches: 1125

Performing warm-up using 3 batches...
Warm-up completed.

Starting MaxViT resource run 1/5
Run 1: 209.04 seconds | 11.6132 ms/image | 86.11 images/s

Starting MaxViT resource run 2/5
Run 2: 239.54 seconds | 13.3079 ms/image | 75.14 images/s

Starting MaxViT resource run 3/5
Run 3: 223.84 seconds | 12.4355 ms/image | 80.41 images/s

Starting MaxViT resource run 4/5
Run 4: 198.58 seconds | 11.0322 ms/image | 90.64 images/s

Starting MaxViT resource run 5/5
Run 5: 183.04 seconds | 10.1689 ms/image | 98.34 images/s

Individual DinoV2 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,209.037759,11.613209,86.108845,2.565274,6.718750,5250.003906,5260.010712,5265.441406,10.006806,15.437500,...,119.803815,120.0,34.215259,100,48.200040,140.544,2.809688,1,18000,-0.405718
1,239.541951,13.307886,75.143414,2.513208,6.300000,5261.191406,5261.927822,5266.710938,0.736416,5.519531,...,119.831461,120.0,33.713483,100,37.904434,134.908,2.532869,2,18000,-0.405718
2,223.839213,12.435512,80.414864,2.557007,7.615625,5262.117188,5204.208371,5266.957031,0.000000,4.839844,...,119.817629,120.0,34.587639,100,41.899255,143.315,2.613988,3,18000,-0.405718
3,198.579511,11.032195,90.643792,2.599557,5.859375,5138.390625,5115.356945,5143.214844,0.000000,4.824219,...,119.787360,120.0,34.300059,79,51.110249,145.018,2.835847,4,18000,-0.405718
4,183.040298,10.168905,98.339001,2.627780,5.859375,4535.746094,4536.038846,4540.675781,0.292752,4.929688,...,119.765013,120.0,34.105744,100,57.520577,148.214,2.927226,5,18000,-0.405718


In [18]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("DinoV2 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


DinoV2 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,210.807746,21.904152,183.610150,238.005343
1,latency_ms_per_image,11.711541,1.216897,10.200564,13.222519
2,throughput_images_per_s,86.129983,8.983350,74.975681,97.284286
3,average_cpu_percent,2.572565,0.043577,2.518458,2.626673
4,peak_cpu_percent,6.470625,0.732966,5.560527,7.380723
5,average_ram_mb,5075.508539,307.397256,4693.824433,5457.192646
6,peak_ram_mb,5096.600000,315.313872,4705.086118,5488.113882
7,average_incremental_ram_mb,2.207195,4.370516,-3.219517,7.633907
8,peak_incremental_ram_mb,7.110156,4.663931,1.319121,12.901192
9,average_gpu_memory_mb,3666.199493,0.025949,3666.167273,3666.231713



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 11.712 ± 1.217 (95% CI: 10.201–13.223)
peak_ram_mb: 5096.600 ± 315.314 (95% CI: 4705.086–5488.114)
peak_gpu_memory_mb: 3666.398 ± 0.000 (95% CI: 3666.398–3666.398)
average_gpu_utilization_percent: 34.184 ± 0.318 (95% CI: 33.789–34.579)
average_gpu_power_w: 47.327 ± 7.701 (95% CI: 37.765–56.889)


genralization

In [20]:
print("\nTest results of wild deepfake dataset on Celeb-DF(V2) (DinoV2):")
test_dataset = DeepfakeDinoV2Dataset(images=test_celeb,labels=test_labels,processor=processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, report, predictions_df = (evaluate_dinov2(model=model,data_loader=test_loader,criterion=criterion,device=device,return_details=True ))
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")


Test results of wild deepfake dataset on Celeb-DF(V2) (DinoV2):

DINOV2-BASE TEST RESULTS
test_loss                     : 0.403020

Confusion Matrix:
accuracy                      : 0.898171

Confusion Matrix:
balanced_accuracy             : 0.498226

Confusion Matrix:
precision                     : 0.903901

Confusion Matrix:
recall_sensitivity            : 0.992950

Confusion Matrix:
specificity                   : 0.003503

Confusion Matrix:
f1_score                      : 0.946335

Confusion Matrix:
mcc                           : -0.012788

Confusion Matrix:
roc_auc                       : 0.521223

Confusion Matrix:
pr_auc                        : 0.913995

Confusion Matrix:
average_precision             : 0.914028

Confusion Matrix:
eer                           : 0.496924

Confusion Matrix:
eer_threshold                 : 0.756497

Confusion Matrix:
false_positive_rate           : 0.996497

Confusion Matrix:
false_negative_rate           : 0.007050

Confusion Matrix:
true_neg

In [22]:
#dfc on wilddeepfake
print("\nTest results of wild deepfake dataset on DFC (DinoV2):")
# Dataloaders
test_dataset = DeepfakeDinoV2Dataset(images=test_hog,labels=test_labels,processor=processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, report, predictions_df = (evaluate_dinov2(model=model,data_loader=test_loader,criterion=criterion,device=device,return_details=True ))
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")


Test results of wild deepfake dataset on DFC (DinoV2):

DINOV2-BASE TEST RESULTS
test_loss                     : 0.876551

Confusion Matrix:
accuracy                      : 0.500000

Confusion Matrix:
balanced_accuracy             : 0.500000

Confusion Matrix:
precision                     : 0.500000

Confusion Matrix:
recall_sensitivity            : 0.999333

Confusion Matrix:
specificity                   : 0.000667

Confusion Matrix:
f1_score                      : 0.666518

Confusion Matrix:
mcc                           : 0.000000

Confusion Matrix:
roc_auc                       : 0.556933

Confusion Matrix:
pr_auc                        : 0.536082

Confusion Matrix:
average_precision             : 0.537258

Confusion Matrix:
eer                           : 0.459667

Confusion Matrix:
eer_threshold                 : 0.780192

Confusion Matrix:
false_positive_rate           : 0.999333

Confusion Matrix:
false_negative_rate           : 0.000667

Confusion Matrix:
true_negatives    

In [24]:
print("\nTest results of wild deepfake dataset on FF++ (DinoV2):")
test_dataset = DeepfakeDinoV2Dataset(images=test_ff,labels=test_ff_labels,processor=processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, report, predictions_df = (evaluate_dinov2(model=model,data_loader=test_loader,criterion=criterion,device=device,return_details=True ))
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")


Test results of wild deepfake dataset on FF++ (DinoV2):

DINOV2-BASE TEST RESULTS
test_loss                     : 0.907952

Confusion Matrix:
accuracy                      : 0.414949

Confusion Matrix:
balanced_accuracy             : 0.496257

Confusion Matrix:
precision                     : 0.414724

Confusion Matrix:
recall_sensitivity            : 0.983511

Confusion Matrix:
specificity                   : 0.009003

Confusion Matrix:
f1_score                      : 0.583429

Confusion Matrix:
mcc                           : -0.033726

Confusion Matrix:
roc_auc                       : 0.505253

Confusion Matrix:
pr_auc                        : 0.428362

Confusion Matrix:
average_precision             : 0.429249

Confusion Matrix:
eer                           : 0.495394

Confusion Matrix:
eer_threshold                 : 0.720600

Confusion Matrix:
false_positive_rate           : 0.990997

Confusion Matrix:
false_negative_rate           : 0.016489

Confusion Matrix:
true_negatives  

# Celeb

In [23]:
import os
import cv2
import numpy as np

SAVE_ROOT = r'D:\thesis\celeb_processed'

def load_split(split_name, class_name):
    """Reload saved frames, grouped by video."""
    base = os.path.join(SAVE_ROOT, split_name, class_name)
    nested, ids = [], []
    for vid_id in sorted(os.listdir(base)):
        vid_dir = os.path.join(base, vid_id)
        frames = [cv2.imread(os.path.join(vid_dir, f))
                  for f in sorted(os.listdir(vid_dir))]
        if frames:
            nested.append(frames)
            ids.append(vid_id)
    return nested, ids

# Reload ALL six splits
print("Loading frames...")
real_train_final,  real_train_ids  = load_split('train', 'real')
synth_train_final, synth_train_ids = load_split('train', 'fake')
real_val_final,    real_val_ids    = load_split('val',   'real')
synth_val_final,   synth_val_ids   = load_split('val',   'fake')
real_test_final,   real_test_ids   = load_split('test',  'real')
synth_test_final,  synth_test_ids  = load_split('test',  'fake')

print("✅ All frames reloaded")
print("Train -> real videos:", len(real_train_final), " fake videos:", len(synth_train_final))
print("Val   -> real videos:", len(real_val_final),   " fake videos:", len(synth_val_final))
print("Test  -> real videos:", len(real_test_final),  " fake videos:", len(synth_test_final))
print("Example frame shape:", np.shape(real_train_final[0][0]))  # expect (160,160,3)
import numpy as np


def combine_split(real_videos, fake_videos):
    """
    Flatten video-grouped frames into one image array and create labels.

    real_videos: list of videos, where each video is a list of frames
    fake_videos: list of videos, where each video is a list of frames

    Returns
    -------
    images : NumPy array with shape (N, 160, 160, 3)
    labels : NumPy array with shape (N,)
             0 = real, 1 = fake
    """

    # Flatten frames from all real videos
    real_frames = [
        frame
        for video_frames in real_videos
        for frame in video_frames
        if frame is not None
    ]

    # Flatten frames from all fake videos
    fake_frames = [
        frame
        for video_frames in fake_videos
        for frame in video_frames
        if frame is not None
    ]

    if len(real_frames) == 0:
        raise ValueError("No real frames were found.")

    if len(fake_frames) == 0:
        raise ValueError("No fake frames were found.")

    # Convert to NumPy arrays
    real_frames = np.stack(real_frames).astype(np.uint8)
    fake_frames = np.stack(fake_frames).astype(np.uint8)

    # Combine images
    images = np.concatenate(
        [real_frames, fake_frames],
        axis=0
    )

    # Create labels
    real_labels = np.zeros(
        len(real_frames),
        dtype=np.uint8
    )

    fake_labels = np.ones(
        len(fake_frames),
        dtype=np.uint8
    )

    labels = np.concatenate(
        [real_labels, fake_labels],
        axis=0
    )

    return images, labels
# Training set
train_celeb, train_labels = combine_split(
    real_train_final,
    synth_train_final
)

# Validation set
val_celeb, val_labels = combine_split(
    real_val_final,
    synth_val_final
)

# Testing set
test_celeb, test_labels = combine_split(
    real_test_final,
    synth_test_final
)
print("\nTRAIN")
print("Images:", train_celeb.shape)
print("Labels:", train_labels.shape)
print("Real:", np.sum(train_labels == 0))
print("Fake:", np.sum(train_labels == 1))

print("\nVALIDATION")
print("Images:", val_celeb.shape)
print("Labels:", val_labels.shape)
print("Real:", np.sum(val_labels == 0))
print("Fake:", np.sum(val_labels == 1))

print("\nTEST")
print("Images:", test_celeb.shape)
print("Labels:", test_labels.shape)
print("Real:", np.sum(test_labels == 0))
print("Fake:", np.sum(test_labels == 1))

print("\nData types")
print("Train images:", train_celeb.dtype)
print("Train labels:", train_labels.dtype)

Loading frames...
✅ All frames reloaded
Train -> real videos: 354  fake videos: 3383
Val   -> real videos: 59  fake videos: 563
Test  -> real videos: 177  fake videos: 1693
Example frame shape: (160, 160, 3)

TRAIN
Images: (11899, 160, 160, 3)
Labels: (11899,)
Real: 1142
Fake: 10757

VALIDATION
Images: (1969, 160, 160, 3)
Labels: (1969,)
Real: 182
Fake: 1787

TEST
Images: (5961, 160, 160, 3)
Labels: (5961,)
Real: 571
Fake: 5390

Data types
Train images: uint8
Train labels: uint8


In [32]:
# ============================================================
# DATASETS
# ============================================================
train_dataset = DeepfakeDinoV2Dataset(images=train_celeb,labels=train_labels,processor=processor)
val_dataset = DeepfakeDinoV2Dataset(images=val_celeb,labels=val_labels,processor=processor)
test_dataset = DeepfakeDinoV2Dataset(images=test_celeb,labels=test_labels,processor=processor)
# ============================================================
# DATALOADERS
# ============================================================
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())


print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training samples: 11899
Validation samples: 1969
Testing samples: 5961
Training batches: 744
Validation batches: 124
Testing batches: 373


In [ ]:
history = train_dinov2(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=EPOCHS
)


Epoch 01/10 | Train Loss: 0.0452 | Train Accuracy: 0.5134 | Val Loss: 0.0428 | Val Accuracy: 0.5422
Epoch 02/10 | Train Loss: 0.0435 | Train Accuracy: 0.5490 | Val Loss: 0.0424 | Val Accuracy: 0.5571
Epoch 03/10 | Train Loss: 0.0429 | Train Accuracy: 0.5622 | Val Loss: 0.0422 | Val Accuracy: 0.5618
Epoch 04/10 | Train Loss: 0.0323 | Train Accuracy: 0.7383 | Val Loss: 0.0275 | Val Accuracy: 0.8082
Epoch 05/10 | Train Loss: 0.0168 | Train Accuracy: 0.9066 | Val Loss: 0.0139 | Val Accuracy: 0.9311
Epoch 06/10 | Train Loss: 0.0133 | Train Accuracy: 0.9348 | Val Loss: 0.0126 | Val Accuracy: 0.9473
Epoch 07/10 | Train Loss: 0.0183 | Train Accuracy: 0.9127 | Val Loss: 0.0295 | Val Accuracy: 0.8481
Epoch 08/10 | Train Loss: 0.0124 | Train Accuracy: 0.9495 | Val Loss: 0.0447 | Val Accuracy: 0.8812
Epoch 09/10 | Train Loss: 0.0090 | Train Accuracy: 0.9695 | Val Loss: 0.0101 | Val Accuracy: 0.9703
Epoch 10/10 | Train Loss: 0.0063 | Train Accuracy: 0.9799 | Val Loss: 0.0081 | Val Accuracy: 0.9757

In [34]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [35]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [ ]:
test_results, confusion, report, predictions_df = (
    evaluate_dinov2(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")
print(confusion)

print("\nConfusion Matrix Format:")
print("[[TN, FP],")
print(" [FN, TP]]")

print("\nClassification Report:")
print(report)



DINOV2-BASE TEST RESULTS
test_loss                     : 0.107990

Confusion Matrix:
accuracy                      : 0.964771

Confusion Matrix:
balanced_accuracy             : 0.841948

Confusion Matrix:
precision                     : 0.968016

Confusion Matrix:
recall_sensitivity            : 0.993878

Confusion Matrix:
specificity                   : 0.690018

Confusion Matrix:
f1_score                      : 0.980776

Confusion Matrix:
mcc                           : 0.780492

Confusion Matrix:
roc_auc                       : 0.980537

Confusion Matrix:
pr_auc                        : 0.997704

Confusion Matrix:
average_precision             : 0.997704

Confusion Matrix:
eer                           : 0.068566

Confusion Matrix:
eer_threshold                 : 0.983558

Confusion Matrix:
false_positive_rate           : 0.309982

Confusion Matrix:
false_negative_rate           : 0.006122

Confusion Matrix:
true_negatives                : 394

Confusion Matrix:
false_positives    

In [37]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 14.0%
Time Usage: 88.4 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [38]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 15.0%
Time Usage: 90.5 s
GPU Memory Used: 2350.4 MB
Power Consumption: 93W


save the model

In [39]:
# ============================================================
# SAVE DINOV2-BASE CLASSIFIER
# ============================================================

import os
import torch


SAVE_DIR = os.path.join(
    r"D:\thesis\results",
    "dinov2_base_celeb_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


WEIGHTS_PATH = os.path.join(
    SAVE_DIR,
    "dinov2_classifier_weights.pth"
)

CHECKPOINT_PATH = os.path.join(
    SAVE_DIR,
    "training_checkpoint.pt"
)


# ============================================================
# SAVE PROCESSOR
# ============================================================

processor.save_pretrained(
    SAVE_DIR
)


# ============================================================
# SAVE MODEL WEIGHTS ONLY
# ============================================================

torch.save(
    model.state_dict(),
    WEIGHTS_PATH
)


# ============================================================
# SAVE COMPLETE TRAINING CHECKPOINT
# ============================================================

completed_epochs = len(
    history.get(
        "train_loss",
        []
    )
)


checkpoint = {
    # Complete DINOv2 backbone and classifier-head weights
    "model_state_dict":
        model.state_dict(),

    # Optimizer state for resuming training
    "optimizer_state_dict":
        optimizer.state_dict(),

    # Training and validation history
    "history":
        history,

    # Model configuration
    "model_name":
        MODEL_NAME,

    "input_size":
        INPUT_SIZE,

    "num_classes":
        NUM_CLASSES,

    "dropout_rate":
        0.3,

    # Training configuration
    "completed_epochs":
        completed_epochs,

    "configured_epochs":
        EPOCHS,

    "batch_size":
        BATCH_SIZE,

    "learning_rate":
        LEARNING_RATE,

    "random_seed":
        RANDOM_SEED,

    # Label interpretation
    "label_mapping": {
        0: "real",
        1: "fake"
    },

    # Processor preprocessing information
    "image_mean":
        processor.image_mean,

    "image_std":
        processor.image_std,

    "processor_size":
        processor.size
}


torch.save(
    checkpoint,
    CHECKPOINT_PATH
)


# ============================================================
# VERIFY SAVED FILES
# ============================================================

print("DINOv2 model saved successfully.")
print("Save directory:", SAVE_DIR)
print("Weights:", WEIGHTS_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

print("\nSaved files:")

for filename in os.listdir(SAVE_DIR):

    file_path = os.path.join(
        SAVE_DIR,
        filename
    )

    if os.path.isfile(file_path):

        file_size_mb = (
            os.path.getsize(file_path)
            / (1024 ** 2)
        )

        print(
            f"{filename:40s}: "
            f"{file_size_mb:.2f} MB"
        )

DINOv2 model saved successfully.
Save directory: D:\thesis\results\dinov2_base_celeb_160
Weights: D:\thesis\results\dinov2_base_celeb_160\dinov2_classifier_weights.pth
Checkpoint: D:\thesis\results\dinov2_base_celeb_160\training_checkpoint.pt

Saved files:
dinov2_classifier_weights.pth           : 330.37 MB
preprocessor_config.json                : 0.00 MB
training_checkpoint.pt                  : 991.11 MB


In [40]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


In [41]:
# ============================================================
# PYTORCH DinoV2 WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        # MaxViT dataset returns:
        # images tensor, labels tensor
        images, labels = batch

        images = images.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # timm MaxViT returns logits directly
        logits = model(images)

        # Materialize part of the output
        _ = logits[-1, 0].item()


if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting DinoV2 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous CUDA operations remain queued
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for images, labels in test_loader:

            images = images.to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            # timm MaxViT forward pass
            logits = model(images)

            last_logits = logits

            processed_images += images.size(0)


    # Wait for all CUDA inference operations
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    if last_logits is None:
        monitor.stop(
            elapsed_seconds=0.0,
            number_of_images=0
        )

        raise RuntimeError(
            "No images were processed during inference."
        )


    last_output_value = float(
        last_logits[-1, 0]
        .detach()
        .cpu()
        .item()
    )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number

    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits
    del images


# ============================================================
# DISPLAY INDIVIDUAL RUNS
# ============================================================

results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual DinoV2 profiling runs:"
)

display(results_df)

Device: cuda
Test images: 5961
Test batches: 373

Performing warm-up using 3 batches...
Warm-up completed.

Starting DinoV2 resource run 1/5
Run 1: 56.35 seconds | 9.4531 ms/image | 105.79 images/s

Starting DinoV2 resource run 2/5
Run 2: 60.54 seconds | 10.1556 ms/image | 98.47 images/s

Starting DinoV2 resource run 3/5
Run 3: 59.41 seconds | 9.9668 ms/image | 100.33 images/s

Starting DinoV2 resource run 4/5
Run 4: 59.83 seconds | 10.0365 ms/image | 99.64 images/s

Starting DinoV2 resource run 5/5
Run 5: 60.78 seconds | 10.1970 ms/image | 98.07 images/s

Individual DinoV2 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,56.349821,9.453082,105.785607,2.703988,6.753125,1970.253906,1975.217236,1979.125000,4.963330,8.871094,...,47.691649,48.0,34.132762,100,58.661257,143.566,0.921515,1,5961,-0.944631
1,60.537252,10.155553,98.468295,2.695448,5.859375,1975.339844,1975.871378,1980.527344,0.531535,5.187500,...,47.716535,48.0,31.856299,93,54.764561,143.669,0.932269,2,5961,-0.944631
2,59.411877,9.966763,100.333474,2.724804,5.771875,1975.773438,1976.059005,1980.671875,0.285567,4.898438,...,47.708502,48.0,32.404858,100,57.602198,142.561,0.952119,3,5961,-0.944631
3,59.827776,10.036533,99.635995,2.702111,6.215625,1976.082031,1976.411330,1981.066406,0.329298,4.984375,...,47.709677,48.0,31.939516,99,56.395192,142.353,0.945398,4,5961,-0.944631
4,60.784106,10.196965,98.068400,2.708784,6.659375,1976.316406,1976.823727,1981.480469,0.507320,5.164062,...,47.716535,48.0,31.927165,100,56.254976,142.901,0.948842,5,5961,-0.944631


In [42]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("DinoV2 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


DinoV2 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,59.382166,1.781448,57.170207,61.594126
1,latency_ms_per_image,9.961779,0.298851,9.590707,10.332851
2,throughput_images_per_s,100.458354,3.112222,96.594020,104.322688
3,average_cpu_percent,2.707027,0.011029,2.693332,2.720722
4,peak_cpu_percent,6.251875,0.448073,5.695519,6.808231
5,average_ram_mb,1976.076535,0.602323,1975.328652,1976.824418
6,peak_ram_mb,1980.574219,0.890998,1979.467899,1981.680538
7,average_incremental_ram_mb,1.323410,2.037613,-1.206620,3.853440
8,peak_incremental_ram_mb,5.821094,1.709314,3.698700,7.943488
9,average_gpu_memory_mb,3946.982017,0.010179,3946.969378,3946.994656



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 9.962 ± 0.299 (95% CI: 9.591–10.333)
peak_ram_mb: 1980.574 ± 0.891 (95% CI: 1979.468–1981.681)
peak_gpu_memory_mb: 3947.273 ± 0.000 (95% CI: 3947.273–3947.273)
average_gpu_utilization_percent: 32.452 ± 0.964 (95% CI: 31.255–33.650)
average_gpu_power_w: 56.736 ± 1.474 (95% CI: 54.905–58.566)


#genralization

In [44]:
#wild deepfake on celeb
print("\nTest results of Celeb-DF(V2) on wild deepfake dataset (MaxViT):")
test_dataset = DeepfakeDinoV2Dataset(images=test_images,labels=test_labels,processor=processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, report, predictions_df = (evaluate_dinov2(model=model,data_loader=test_loader,criterion=criterion,device=device,return_details=True ))
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")


Test results of Celeb-DF(V2) on wild deepfake dataset (MaxViT):

DINOV2-BASE TEST RESULTS
test_loss                     : 0.678271

Confusion Matrix:
accuracy                      : 0.750000

Confusion Matrix:
balanced_accuracy             : 0.500000

Confusion Matrix:
precision                     : 0.750000

Confusion Matrix:
recall_sensitivity            : 1.000000

Confusion Matrix:
specificity                   : 0.000000

Confusion Matrix:
f1_score                      : 0.857143

Confusion Matrix:
mcc                           : 0.000000

Confusion Matrix:
roc_auc                       : 0.506252

Confusion Matrix:
pr_auc                        : 0.777892

Confusion Matrix:
average_precision             : 0.777909

Confusion Matrix:
eer                           : 0.471889

Confusion Matrix:
eer_threshold                 : 0.912675

Confusion Matrix:
false_positive_rate           : 1.000000

Confusion Matrix:
false_negative_rate           : 0.000000

Confusion Matrix:
true_nega

In [46]:
#DFC on celeb
print("\nTest results of Celeb-DF(V2) on DFC dataset (DinoV2):")
test_dataset = DeepfakeDinoV2Dataset(images=test_hog,labels=test_labels,processor=processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, report, predictions_df = (evaluate_dinov2(model=model,data_loader=test_loader,criterion=criterion,device=device,return_details=True ))
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")


Test results of Celeb-DF(V2) on DFC dataset (DinoV2):

DINOV2-BASE TEST RESULTS
test_loss                     : 1.222704

Confusion Matrix:
accuracy                      : 0.500000

Confusion Matrix:
balanced_accuracy             : 0.500000

Confusion Matrix:
precision                     : 0.500000

Confusion Matrix:
recall_sensitivity            : 1.000000

Confusion Matrix:
specificity                   : 0.000000

Confusion Matrix:
f1_score                      : 0.666667

Confusion Matrix:
mcc                           : 0.000000

Confusion Matrix:
roc_auc                       : 0.402975

Confusion Matrix:
pr_auc                        : 0.434484

Confusion Matrix:
average_precision             : 0.435429

Confusion Matrix:
eer                           : 0.572333

Confusion Matrix:
eer_threshold                 : 0.905248

Confusion Matrix:
false_positive_rate           : 1.000000

Confusion Matrix:
false_negative_rate           : 0.000000

Confusion Matrix:
true_negatives     

In [48]:
print("\nTest results of wild deepfake dataset on FF++ (DinoV2):")
#ff++ on wilddeepfake
test_dataset = DeepfakeDinoV2Dataset(images=test_ff,labels=test_ff_labels,processor=processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, report, predictions_df = (evaluate_dinov2(model=model,data_loader=test_loader,criterion=criterion,device=device,return_details=True ))
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")
    test_results_df = pd.DataFrame(
    [test_results]
)

display(test_results_df)
display(predictions_df.head())


Test results of wild deepfake dataset on FF++ (DinoV2):

DINOV2-BASE TEST RESULTS
test_loss                     : 1.453563

Confusion Matrix:
accuracy                      : 0.416566

Confusion Matrix:
balanced_accuracy             : 0.500000

Confusion Matrix:
precision                     : 0.416566

Confusion Matrix:
recall_sensitivity            : 1.000000

Confusion Matrix:
specificity                   : 0.000000

Confusion Matrix:
f1_score                      : 0.588135

Confusion Matrix:
mcc                           : 0.000000

Confusion Matrix:
roc_auc                       : 0.587806

Confusion Matrix:
pr_auc                        : 0.480860

Confusion Matrix:
average_precision             : 0.481900

Confusion Matrix:
eer                           : 0.434370

Confusion Matrix:
eer_threshold                 : 0.912685

Confusion Matrix:
false_positive_rate           : 1.000000

Confusion Matrix:
false_negative_rate           : 0.000000

Confusion Matrix:
true_negatives   

,test_loss,accuracy,balanced_accuracy,precision,recall_sensitivity,specificity,f1_score,mcc,roc_auc,pr_auc,average_precision,eer,eer_threshold,false_positive_rate,false_negative_rate,true_negatives,false_positives,false_negatives,true_positives,number_of_test_images
0,1.453563,0.416566,0.5,0.416566,1.0,0.0,0.588135,0.0,0.587806,0.48086,0.4819,0.43437,0.912685,1.0,0.0,0,1444,0,1031,2475


,true_label,predicted_label,fake_probability
0,0,1,0.917542
1,0,1,0.916013
2,0,1,0.917262
3,0,1,0.913442
4,0,1,0.913862


# DFC

In [25]:
import h5py
import numpy as np
# Open the HDF5 file in read mode
with h5py.File('D://thesis//dataset//deepfake dataset//resized_images.h5', 'r') as h5f:
    # Access each dataset
    celeb = np.array(h5f['celeb'])
    ffhq = np.array(h5f['ffhq'])
    gdwct = np.array(h5f['gdwct'])
    attgan = np.array(h5f['attgan'])
    stargan = np.array(h5f['stargan'])
    stylegan2 = np.array(h5f['stylegan2'])
    stylegan = np.array(h5f['stylegan'])

# Now, 'celeb', 'ffhq', etc., are NumPy arrays containing your datasets
print(f"celeb shape: {celeb.shape}, dtype: {celeb.dtype}")
print(f"ffhq shape: {ffhq.shape}, dtype: {ffhq.dtype}")
print(f"ffhq shape: {gdwct.shape}, dtype: {gdwct.dtype}")
print(f"ffhq shape: {attgan.shape}, dtype: {attgan.dtype}")
print(f"ffhq shape: {stargan.shape}, dtype: {stargan.dtype}")
print(f"ffhq shape: {stylegan2.shape}, dtype: {stylegan2.dtype}")
print(f"ffhq shape: {stylegan.shape}, dtype: {stylegan.dtype}")
# Repeat for other datasets as needed
import cv2
# Function to resize images from (224, 224) to (160, 160)
def resize_images(image_array, target_size=(160, 160)):
    resized_images = np.array([cv2.resize(img, target_size) for img in image_array])
    return resized_images

celeb = resize_images(celeb, target_size=(160, 160))
ffhq = resize_images(ffhq, target_size=(160, 160))
gdwct = resize_images(gdwct, target_size=(160, 160))
attgan = resize_images(attgan, target_size=(160, 160))
stargan = resize_images(stargan, target_size=(160, 160))
stylegan = resize_images(stylegan, target_size=(160, 160))
stylegan2 = resize_images(stylegan2, target_size=(160, 160))
import random
# Randomly select 2500 distinct images
random_indices = random.sample(range(len(celeb)), 2500)  # Get 2500 random indices
celeb = celeb[random_indices]  # Select the random subse

import random
# Randomly select 2500 distinct images
random_indices = random.sample(range(len(ffhq)), 2500)  # Get 2500 random indices
ffhq = ffhq[random_indices]  # Select the random subse
print(f"celeb shape: {celeb.shape}, dtype: {celeb.dtype}")
print(f"ffhq shape: {ffhq.shape}, dtype: {ffhq.dtype}")
print(f"gdwct shape: {gdwct.shape}, dtype: {gdwct.dtype}")
print(f"attagan shape: {attgan.shape}, dtype: {attgan.dtype}")
print(f"stargan shape: {stargan.shape}, dtype: {stargan.dtype}")
print(f"stylegan2 shape: {stylegan2.shape}, dtype: {stylegan2.dtype}")
print(f"stylegan shape: {stylegan.shape}, dtype: {stylegan.dtype}")
import random
import numpy as np

def split_data(data, train_ratio=0.7):
    """
    Splits data into training and testing sets based on the specified ratio.

    Parameters:
        data (list or np.array): The dataset to split.
        train_ratio (float): The ratio of the data to include in the training set.

    Returns:
        tuple: Two datasets - train and test.
    """
    # Shuffle the data
    random.shuffle(data)

    # Calculate the split index
    split_index = int(len(data) * train_ratio)

    # Split the data
    train_data = data[:split_index]
    test_data = data[split_index:]

    return train_data, test_data

# Split `celeb` into 70% train and 30% test
celeb_train_hog, celeb_test_hog = split_data(celeb, train_ratio=0.7)

# Split `ffhq` into 70% train and 30% test
ffhq_train_hog, ffhq_test_hog = split_data(ffhq, train_ratio=0.7)

# Split `attgan` into 70% train and 30% test
attgan_train_hog, attgan_test_hog = split_data(attgan, train_ratio=0.7)

# Split `stargan` into 70% train and 30% test
stargan_train_hog, stargan_test_hog = split_data(stargan, train_ratio=0.7)

# Split `gdwct` into 70% train and 30% test
gdwct_train_hog, gdwct_test_hog = split_data(gdwct, train_ratio=0.7)

# Split `stylegan2` into 70% train and 30% test_hog
stylegan2_train_hog, stylegan2_test_hog = split_data(stylegan2, train_ratio=0.7)

# Split `stylegan` into 70% train and 30% test_hog
stylegan_train_hog, stylegan_test_hog = split_data(stylegan, train_ratio=0.7)

# Convert to NumPy arrays if needed
celeb_train_hog, celeb_test_hog = np.array(celeb_train_hog), np.array(celeb_test_hog)
ffhq_train_hog, ffhq_test_hog = np.array(ffhq_train_hog), np.array(ffhq_test_hog)
attgan_train_hog, attgan_test_hog = np.array(attgan_train_hog), np.array(attgan_test_hog)
stargan_train_hog, stargan_test_hog = np.array(stargan_train_hog), np.array(stargan_test_hog)
gdwct_train_hog, gdwct_test_hog = np.array(gdwct_train_hog), np.array(gdwct_test_hog)
stylegan2_train_hog, stylegan2_test_hog = np.array(stylegan2_train_hog), np.array(stylegan2_test_hog)
stylegan_train_hog, stylegan_test_hog = np.array(stylegan_train_hog), np.array(stylegan_test_hog)

# Print results for verification
print(f"celeb_train: {len(celeb_train_hog)} images, celeb_test: {len(celeb_test_hog)} images")
print(f"ffhq_train: {len(ffhq_train_hog)} images, ffhq_test: {len(ffhq_test_hog)} images")
print(f"attgan_train: {len(attgan_train_hog)} images, attgan_test: {len(attgan_test_hog)} images")
print(f"stargan_train: {len(stargan_train_hog)} images, stargan_test: {len(stargan_test_hog)} images")
print(f"gdwct_train: {len(gdwct_train_hog)} images, gdwct_test: {len(gdwct_test_hog)} images")
print(f"stylegan2_train: {len(stylegan2_train_hog)} images, stylegan2_test: {len(stylegan2_test_hog)} images")
print(f"stylegan_train: {len(stylegan_train_hog)} images, stylegan_test: {len(stylegan_test_hog)} images")

########################################################################################################################################
#######################################divide into 60,10 train and val
#########################################################################################################################################
def extract_validation(train_data):
    """
    Extract every 10th sample from the training data and store it in a validation set.

    Parameters:
        train_data (list or np.array): The training dataset.

    Returns:
        tuple: Updated training dataset and validation dataset.
    """
    # Select every 10th sample for the validation set
    validation_data = train_data[::10]

    # Remove the selected samples from the training dataset
    updated_train_data = [train_data[i] for i in range(len(train_data)) if i % 10 != 0]

    return np.array(updated_train_data), np.array(validation_data)


# Perform the operation for each dataset
celeb_train_hog, celeb_val_hog = extract_validation(celeb_train_hog)
ffhq_train_hog, ffhq_val_hog = extract_validation(ffhq_train_hog)
attgan_train_hog, attgan_val_hog = extract_validation(attgan_train_hog)
stargan_train_hog, stargan_val_hog = extract_validation(stargan_train_hog)
gdwct_train_hog, gdwct_val_hog = extract_validation(gdwct_train_hog)
stylegan2_train_hog, stylegan2_val_hog = extract_validation(stylegan2_train_hog)
stylegan_train_hog, stylegan_val_hog = extract_validation(stylegan_train_hog)

# Print results for verification
print(f"celeb_train: {len(celeb_train_hog)} images, celeb_val: {len(celeb_val_hog)} images")
print(f"ffhq_train: {len(ffhq_train_hog)} images, ffhq_val: {len(ffhq_val_hog)} images")
print(f"attgan_train: {len(attgan_train_hog)} images, attgan_val: {len(attgan_val_hog)} images")
print(f"stargan_train: {len(stargan_train_hog)} images, stargan_val: {len(stargan_val_hog)} images")
print(f"gdwct_train: {len(gdwct_train_hog)} images, gdwct_val: {len(gdwct_val_hog)} images")
print(f"stylegan2_train: {len(stylegan2_train_hog)} images, stylegan2_val: {len(stylegan2_val_hog)} images")
print(f"stylegan_train: {len(stylegan_train_hog)} images, stylegan_val: {len(stylegan_val_hog)} images")
############################################################################################################################################################
#################################################concatenate the labels 0,1 real and fake
#############################################################################################################################################################


celeb_train_labels = np.zeros(len(celeb_train_hog), dtype=int)
ffhq_train_labels = np.zeros(len(ffhq_train_hog), dtype=int)
atta_train_labels = np.ones(len(attgan_train_hog), dtype=int)
star_train_labels = np.ones(len(stargan_train_hog), dtype=int)
gdwct_train_labels = np.ones(len(gdwct_train_hog), dtype=int)
stylegan2_train_labels = np.ones(len(stylegan2_train_hog), dtype=int)
stylegan_train_labels = np.ones(len(stylegan_train_hog), dtype=int)

# Concatenate all training datasets into a single `train` variable
train_hog = np.concatenate([celeb_train_hog, ffhq_train_hog, attgan_train_hog, stargan_train_hog, gdwct_train_hog, stylegan2_train_hog, stylegan_train_hog], axis=0)
train_labels=np.concatenate([celeb_train_labels, ffhq_train_labels, atta_train_labels, star_train_labels, gdwct_train_labels, stylegan2_train_labels,
                              stylegan_train_labels], axis=0)




celeb_test_labels = np.zeros(len(celeb_test_hog), dtype=int)
ffhq_test_labels = np.zeros(len(ffhq_test_hog), dtype=int)
atta_test_labels = np.ones(len(attgan_test_hog), dtype=int)
star_test_labels = np.ones(len(stargan_test_hog), dtype=int)
gdwct_test_labels = np.ones(len(gdwct_test_hog), dtype=int)
stylegan2_test_labels = np.ones(len(stylegan2_test_hog), dtype=int)
stylegan_test_labels = np.ones(len(stylegan_test_hog), dtype=int)

# Concatenate all testing datasets into a single `test` variable
test_hog = np.concatenate([celeb_test_hog, ffhq_test_hog, attgan_test_hog, stargan_test_hog, gdwct_test_hog, stylegan2_test_hog, stylegan_test_hog], axis=0)
test_labels = np.concatenate([celeb_test_labels, ffhq_test_labels, atta_test_labels, star_test_labels, gdwct_test_labels, stylegan2_test_labels,
                        stylegan_test_labels], axis=0)




celeb_val_labels = np.zeros(len(celeb_val_hog), dtype=int)
ffhq_val_labels = np.zeros(len(ffhq_val_hog), dtype=int)
atta_val_labels = np.ones(len(attgan_val_hog), dtype=int)
star_val_labels = np.ones(len(stargan_val_hog), dtype=int)
gdwct_val_labels = np.ones(len(gdwct_val_hog), dtype=int)
stylegan2_val_labels = np.ones(len(stylegan2_val_hog), dtype=int)
stylegan_val_labels = np.ones(len(stylegan_val_hog), dtype=int)

# Concatenate all validation datasets into a single `val` variable
val_hog = np.concatenate([celeb_val_hog, ffhq_val_hog, attgan_val_hog, stargan_val_hog, gdwct_val_hog, stylegan2_val_hog, stylegan_val_hog], axis=0)
val_labels = np.concatenate([celeb_val_labels, ffhq_val_labels, atta_val_labels, star_val_labels, gdwct_val_labels, stylegan2_val_labels,
                       stylegan_val_labels], axis=0)

# Print the results for verification
print(f"Total train: {len(train_hog)} images")
print(f"Total test: {len(test_hog)} images")
print(f"Total val: {len(val_hog)} images")


# Print results for verification
print(f"Train Labels: {len(train_labels)} ")
print(f"Test Labels: {len(test_labels)} ")
print(f"Val Labels: {len(val_labels)} ")



celeb shape: (5000, 224, 224, 3), dtype: uint8
ffhq shape: (5000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
celeb shape: (2500, 160, 160, 3), dtype: uint8
ffhq shape: (2500, 160, 160, 3), dtype: uint8
gdwct shape: (1000, 160, 160, 3), dtype: uint8
attagan shape: (1000, 160, 160, 3), dtype: uint8
stargan shape: (1000, 160, 160, 3), dtype: uint8
stylegan2 shape: (1000, 160, 160, 3), dtype: uint8
stylegan shape: (1000, 160, 160, 3), dtype: uint8
celeb_train: 1750 images, celeb_test: 750 images
ffhq_train: 1750 images, ffhq_test: 750 images
attgan_train: 700 images, attgan_test: 300 images
stargan_train: 700 images, stargan_test: 300 images
gdwct_train: 700 images, gdwct_test: 300 images
stylegan2_train: 700 images, stylegan2_test: 300 images
stylegan_train: 700 images, stylegan

In [9]:
# ============================================================
# DATASETS
# ============================================================
train_dataset = DeepfakeDinoV2Dataset(images=train_hog,labels=train_labels,processor=processor)
val_dataset = DeepfakeDinoV2Dataset(images=val_hog,labels=val_labels,processor=processor)
test_dataset = DeepfakeDinoV2Dataset(images=test_hog,labels=test_labels,processor=processor)
# ============================================================
# DATALOADERS
# ============================================================
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())


print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training samples: 6300
Validation samples: 700
Testing samples: 3000
Training batches: 394
Validation batches: 44
Testing batches: 188


In [1]:
history = train_dinov2(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=EPOCHS
)


Epoch 01/10 | Train Loss: 0.0450 | Train Accuracy: 0.5097 | Val Loss: 0.0419 | Val Accuracy: 0.6257
Epoch 02/10 | Train Loss: 0.0431 | Train Accuracy: 0.5597 | Val Loss: 0.0403 | Val Accuracy: 0.6929
Epoch 03/10 | Train Loss: 0.0416 | Train Accuracy: 0.5987 | Val Loss: 0.0389 | Val Accuracy: 0.7186
Epoch 04/10 | Train Loss: 0.0269 | Train Accuracy: 0.8035 | Val Loss: 0.0093 | Val Accuracy: 0.9543
Epoch 05/10 | Train Loss: 0.0075 | Train Accuracy: 0.9657 | Val Loss: 0.0055 | Val Accuracy: 0.9800
Epoch 06/10 | Train Loss: 0.0036 | Train Accuracy: 0.9878 | Val Loss: 0.0031 | Val Accuracy: 0.9914
Epoch 07/10 | Train Loss: 0.0063 | Train Accuracy: 0.9798 | Val Loss: 0.0058 | Val Accuracy: 0.9829
Epoch 08/10 | Train Loss: 0.0025 | Train Accuracy: 0.9919 | Val Loss: 0.0019 | Val Accuracy: 0.9957
Epoch 09/10 | Train Loss: 0.0018 | Train Accuracy: 0.9949 | Val Loss: 0.0056 | Val Accuracy: 0.9857
Epoch 10/10 | Train Loss: 0.0021 | Train Accuracy: 0.9949 | Val Loss: 0.0018 | Val Accuracy: 0.9957

In [15]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [16]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [2]:
test_results, confusion, report, predictions_df = (
    evaluate_dinov2(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")
print(confusion)

print("\nConfusion Matrix Format:")
print("[[TN, FP],")
print(" [FN, TP]]")

print("\nClassification Report:")
print(report)


DINOV2-BASE TEST RESULTS
test_loss                     : 0.086196

Confusion Matrix:
accuracy                      : 0.981000

Confusion Matrix:
balanced_accuracy             : 0.981000

Confusion Matrix:
precision                     : 0.975610

Confusion Matrix:
recall_sensitivity            : 0.986667

Confusion Matrix:
specificity                   : 0.975333

Confusion Matrix:
f1_score                      : 0.981107

Confusion Matrix:
mcc                           : 0.962062

Confusion Matrix:
roc_auc                       : 0.998157

Confusion Matrix:
pr_auc                        : 0.998108

Confusion Matrix:
average_precision             : 0.998107

Confusion Matrix:
eer                           : 0.021000

Confusion Matrix:
eer_threshold                 : 0.855206

Confusion Matrix:
false_positive_rate           : 0.024667

Confusion Matrix:
false_negative_rate           : 0.013333

Confusion Matrix:
true_negatives                : 1463

Confusion Matrix:
false_positives   

In [18]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 13.0%
Time Usage: 46.9 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [19]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 13.5%
Time Usage: 49.0 s
GPU Memory Used: 1349.5 MB
Power Consumption: 93W


save the model

In [20]:
# ============================================================
# SAVE DINOV2-BASE CLASSIFIER
# ============================================================

import os
import torch


SAVE_DIR = os.path.join(
    r"D:\thesis\results",
    "dinov2_base_dfc_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


WEIGHTS_PATH = os.path.join(
    SAVE_DIR,
    "dinov2_classifier_weights.pth"
)

CHECKPOINT_PATH = os.path.join(
    SAVE_DIR,
    "training_checkpoint.pt"
)


# ============================================================
# SAVE PROCESSOR
# ============================================================

processor.save_pretrained(
    SAVE_DIR
)


# ============================================================
# SAVE MODEL WEIGHTS ONLY
# ============================================================

torch.save(
    model.state_dict(),
    WEIGHTS_PATH
)


# ============================================================
# SAVE COMPLETE TRAINING CHECKPOINT
# ============================================================

completed_epochs = len(
    history.get(
        "train_loss",
        []
    )
)


checkpoint = {
    # Complete DINOv2 backbone and classifier-head weights
    "model_state_dict":
        model.state_dict(),

    # Optimizer state for resuming training
    "optimizer_state_dict":
        optimizer.state_dict(),

    # Training and validation history
    "history":
        history,

    # Model configuration
    "model_name":
        MODEL_NAME,

    "input_size":
        INPUT_SIZE,

    "num_classes":
        NUM_CLASSES,

    "dropout_rate":
        0.3,

    # Training configuration
    "completed_epochs":
        completed_epochs,

    "configured_epochs":
        EPOCHS,

    "batch_size":
        BATCH_SIZE,

    "learning_rate":
        LEARNING_RATE,

    "random_seed":
        RANDOM_SEED,

    # Label interpretation
    "label_mapping": {
        0: "real",
        1: "fake"
    },

    # Processor preprocessing information
    "image_mean":
        processor.image_mean,

    "image_std":
        processor.image_std,

    "processor_size":
        processor.size
}


torch.save(
    checkpoint,
    CHECKPOINT_PATH
)


# ============================================================
# VERIFY SAVED FILES
# ============================================================

print("DINOv2 model saved successfully.")
print("Save directory:", SAVE_DIR)
print("Weights:", WEIGHTS_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

print("\nSaved files:")

for filename in os.listdir(SAVE_DIR):

    file_path = os.path.join(
        SAVE_DIR,
        filename
    )

    if os.path.isfile(file_path):

        file_size_mb = (
            os.path.getsize(file_path)
            / (1024 ** 2)
        )

        print(
            f"{filename:40s}: "
            f"{file_size_mb:.2f} MB"
        )

DINOv2 model saved successfully.
Save directory: D:\thesis\results\dinov2_base_dfc_160
Weights: D:\thesis\results\dinov2_base_dfc_160\dinov2_classifier_weights.pth
Checkpoint: D:\thesis\results\dinov2_base_dfc_160\training_checkpoint.pt

Saved files:
dinov2_classifier_weights.pth           : 330.37 MB
preprocessor_config.json                : 0.00 MB
training_checkpoint.pt                  : 991.11 MB


#load the model

In [21]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_34880\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [22]:
# ============================================================
# PYTORCH MAXVIT WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        # MaxViT dataset returns:
        # images tensor, labels tensor
        images, labels = batch

        images = images.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # timm MaxViT returns logits directly
        logits = model(images)

        # Materialize part of the output
        _ = logits[-1, 0].item()


if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting DinoV2 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous CUDA operations remain queued
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for images, labels in test_loader:

            images = images.to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            # timm MaxViT forward pass
            logits = model(images)

            last_logits = logits

            processed_images += images.size(0)


    # Wait for all CUDA inference operations
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    if last_logits is None:
        monitor.stop(
            elapsed_seconds=0.0,
            number_of_images=0
        )

        raise RuntimeError(
            "No images were processed during inference."
        )


    last_output_value = float(
        last_logits[-1, 0]
        .detach()
        .cpu()
        .item()
    )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number

    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits
    del images


# ============================================================
# DISPLAY INDIVIDUAL RUNS
# ============================================================

results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual DinoV2 profiling runs:"
)

display(results_df)

Device: cuda
Test images: 3000
Test batches: 188

Performing warm-up using 3 batches...
Warm-up completed.

Starting DinoV2 resource run 1/5
Run 1: 29.88 seconds | 9.9591 ms/image | 100.41 images/s

Starting DinoV2 resource run 2/5
Run 2: 29.69 seconds | 9.8982 ms/image | 101.03 images/s

Starting DinoV2 resource run 3/5
Run 3: 30.47 seconds | 10.1575 ms/image | 98.45 images/s

Starting DinoV2 resource run 4/5
Run 4: 27.85 seconds | 9.2823 ms/image | 107.73 images/s

Starting DinoV2 resource run 5/5
Run 5: 27.74 seconds | 9.2472 ms/image | 108.14 images/s

Individual DinoV2 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,29.877371,9.959124,100.410442,2.648028,5.468750,6809.593750,6817.992312,6822.488281,8.398562,12.894531,...,118.571429,120.0,33.726190,100,57.449992,142.499,0.485566,1,3000,-1.562424
1,29.694569,9.898190,101.028575,2.634734,5.771875,6818.523438,6818.699843,6821.949219,0.176406,3.425781,...,118.524590,120.0,33.065574,100,55.159619,142.649,0.460731,2,3000,-1.562424
2,30.472511,10.157504,98.449386,2.548970,5.375000,6818.589844,6818.767859,6823.386719,0.178016,4.796875,...,118.636364,120.0,35.181818,100,55.573193,142.632,0.473630,3,3000,-1.562424
3,27.846997,9.282332,107.731545,2.524023,5.078125,6818.597656,6818.723779,6823.394531,0.126123,4.796875,...,118.500000,120.0,40.125000,100,59.488667,134.606,0.463642,4,3000,-1.562424
4,27.741594,9.247198,108.140865,2.515299,5.468750,6818.609375,6818.723128,6821.820312,0.113753,3.210938,...,118.500000,120.0,39.291667,100,58.376296,142.507,0.452408,5,3000,-1.562424


In [23]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("DinoV2 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


DinoV2 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,29.126608,1.250335,27.574113,30.679104
1,latency_ms_per_image,9.708869,0.416778,9.191371,10.226368
2,throughput_images_per_s,103.152163,4.472154,97.599249,108.705076
3,average_cpu_percent,2.574211,0.062726,2.496326,2.652096
4,peak_cpu_percent,5.432500,0.248340,5.124145,5.740855
5,average_ram_mb,6818.581384,0.330220,6818.171363,6818.991406
6,peak_ram_mb,6822.607813,0.757267,6821.667542,6823.548083
7,average_incremental_ram_mb,1.798572,3.689620,-2.782697,6.379840
8,peak_incremental_ram_mb,5.825000,4.021254,0.831954,10.818046
9,average_gpu_memory_mb,3666.983976,0.058098,3666.911838,3667.056114



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 9.709 ± 0.417 (95% CI: 9.191–10.226)
peak_ram_mb: 6822.608 ± 0.757 (95% CI: 6821.668–6823.548)
peak_gpu_memory_mb: 3668.438 ± 0.000 (95% CI: 3668.438–3668.438)
average_gpu_utilization_percent: 36.278 ± 3.237 (95% CI: 32.259–40.297)
average_gpu_power_w: 57.210 ± 1.837 (95% CI: 54.929–59.490)


#genralization

In [25]:
#wild deepfake on dfc
print("\nTest results of DFC on wild deepfake dataset (dINOv2):")
test_dataset = DeepfakeDinoV2Dataset(images=test_images,labels=test_labels,processor=processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, report, predictions_df = (evaluate_dinov2(model=model,data_loader=test_loader,criterion=criterion,device=device,return_details=True ))
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")
    test_results_df = pd.DataFrame(
    [test_results]
)

display(test_results_df)
display(predictions_df.head())


Test results of DFC on wild deepfake dataset (dINOv2):

DINOV2-BASE TEST RESULTS
test_loss                     : 0.793130

Confusion Matrix:
accuracy                      : 0.640722

Confusion Matrix:
balanced_accuracy             : 0.520259

Confusion Matrix:
precision                     : 0.760115

Confusion Matrix:
recall_sensitivity            : 0.761185

Confusion Matrix:
specificity                   : 0.279333

Confusion Matrix:
f1_score                      : 0.760650

Confusion Matrix:
mcc                           : 0.040576

Confusion Matrix:
roc_auc                       : 0.552465

Confusion Matrix:
pr_auc                        : 0.783803

Confusion Matrix:
average_precision             : 0.783872

Confusion Matrix:
eer                           : 0.468444

Confusion Matrix:
eer_threshold                 : 0.839148

Confusion Matrix:
false_positive_rate           : 0.720667

Confusion Matrix:
false_negative_rate           : 0.238815

Confusion Matrix:
true_negatives    

,test_loss,accuracy,balanced_accuracy,precision,recall_sensitivity,specificity,f1_score,mcc,roc_auc,pr_auc,average_precision,eer,eer_threshold,false_positive_rate,false_negative_rate,true_negatives,false_positives,false_negatives,true_positives,number_of_test_images
0,0.79313,0.640722,0.520259,0.760115,0.761185,0.279333,0.76065,0.040576,0.552465,0.783803,0.783872,0.468444,0.839148,0.720667,0.238815,1257,3243,3224,10276,18000


,true_label,predicted_label,fake_probability
0,1,1,0.967469
1,1,1,0.972245
2,1,1,0.981141
3,1,1,0.989907
4,1,1,0.971371


In [27]:
#celeb on dfc
print("\nTest results of DFC on Celeb-DF(V2) dataset (DinoV2):")
test_dataset = DeepfakeDinoV2Dataset(images=test_celeb,labels=test_labels,processor=processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, report, predictions_df = (evaluate_dinov2(model=model,data_loader=test_loader,criterion=criterion,device=device,return_details=True ))
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")
    test_results_df = pd.DataFrame(
    [test_results]
)

display(test_results_df)
display(predictions_df.head())


Test results of DFC on Celeb-DF(V2) dataset (DinoV2):

DINOV2-BASE TEST RESULTS
test_loss                     : 0.619750

Confusion Matrix:
accuracy                      : 0.695185

Confusion Matrix:
balanced_accuracy             : 0.517507

Confusion Matrix:
precision                     : 0.908343

Confusion Matrix:
recall_sensitivity            : 0.737291

Confusion Matrix:
specificity                   : 0.297723

Confusion Matrix:
f1_score                      : 0.813927

Confusion Matrix:
mcc                           : 0.023320

Confusion Matrix:
roc_auc                       : 0.496721

Confusion Matrix:
pr_auc                        : 0.897024

Confusion Matrix:
average_precision             : 0.897189

Confusion Matrix:
eer                           : 0.502241

Confusion Matrix:
eer_threshold                 : 0.788915

Confusion Matrix:
false_positive_rate           : 0.702277

Confusion Matrix:
false_negative_rate           : 0.262709

Confusion Matrix:
true_negatives     

,test_loss,accuracy,balanced_accuracy,precision,recall_sensitivity,specificity,f1_score,mcc,roc_auc,pr_auc,average_precision,eer,eer_threshold,false_positive_rate,false_negative_rate,true_negatives,false_positives,false_negatives,true_positives,number_of_test_images
0,0.61975,0.695185,0.517507,0.908343,0.737291,0.297723,0.813927,0.02332,0.496721,0.897024,0.897189,0.502241,0.788915,0.702277,0.262709,170,401,1416,3974,5961


,true_label,predicted_label,fake_probability
0,0,0,0.317393
1,0,0,0.373945
2,0,0,0.380519
3,0,0,0.207856
4,0,1,0.989325


In [29]:
print("\nTest results of DFC dataset on FF++ (DinoV2):")
test_dataset = DeepfakeDinoV2Dataset(images=test_ff,labels=test_ff_labels,processor=processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, report, predictions_df = (evaluate_dinov2(model=model,data_loader=test_loader,criterion=criterion,device=device,return_details=True ))
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")
    test_results_df = pd.DataFrame(
    [test_results]
)

display(test_results_df)
display(predictions_df.head())


Test results of DFC dataset on FF++ (DinoV2):

DINOV2-BASE TEST RESULTS
test_loss                     : 1.801737

Confusion Matrix:
accuracy                      : 0.412929

Confusion Matrix:
balanced_accuracy             : 0.464981

Confusion Matrix:
precision                     : 0.395751

Confusion Matrix:
recall_sensitivity            : 0.776916

Confusion Matrix:
specificity                   : 0.153047

Confusion Matrix:
f1_score                      : 0.524386

Confusion Matrix:
mcc                           : -0.089443

Confusion Matrix:
roc_auc                       : 0.399884

Confusion Matrix:
pr_auc                        : 0.347984

Confusion Matrix:
average_precision             : 0.348704

Confusion Matrix:
eer                           : 0.580176

Confusion Matrix:
eer_threshold                 : 0.910852

Confusion Matrix:
false_positive_rate           : 0.846953

Confusion Matrix:
false_negative_rate           : 0.223084

Confusion Matrix:
true_negatives            

,test_loss,accuracy,balanced_accuracy,precision,recall_sensitivity,specificity,f1_score,mcc,roc_auc,pr_auc,average_precision,eer,eer_threshold,false_positive_rate,false_negative_rate,true_negatives,false_positives,false_negatives,true_positives,number_of_test_images
0,1.801737,0.412929,0.464981,0.395751,0.776916,0.153047,0.524386,-0.089443,0.399884,0.347984,0.348704,0.580176,0.910852,0.846953,0.223084,221,1223,230,801,2475


,true_label,predicted_label,fake_probability
0,0,0,0.187583
1,0,0,0.115327
2,0,0,0.074920
3,0,1,0.895759
4,0,1,0.910951


# FF++

LOAD THE DATASET

In [7]:
import os, cv2, numpy as np

FINAL_ROOT = r'D:\thesis\ff_final'

def load_split(split, cls):
    base = os.path.join(FINAL_ROOT, split, cls)
    nested, ids = [], []
    for vid_id in sorted(os.listdir(base)):
        d = os.path.join(base, vid_id)
        frames = [cv2.imread(os.path.join(d, f)) for f in sorted(os.listdir(d))]
        if frames:
            nested.append(frames); ids.append(vid_id)
    return nested, ids

# Main splits
ff_real_train_f, ff_real_train_ids = load_split('train', 'real')
ff_fake_train_f, ff_fake_train_ids = load_split('train', 'fake')
ff_real_val_f,   ff_real_val_ids   = load_split('val',   'real')
ff_fake_val_f,   ff_fake_val_ids   = load_split('val',   'fake')
ff_real_test_f,  ff_real_test_ids  = load_split('test',  'real')
ff_fake_test_f,  ff_fake_test_ids  = load_split('test',  'fake')

print("Reloaded main splits. Example shape:", np.shape(ff_real_train_f[0][0]))  # (160,160,3)
print("Real train videos:", len(ff_real_train_f), "| Fake train videos:", len(ff_fake_train_f))
import numpy as np


def combine_split(real_videos, fake_videos):
    """
    Combine all frames from the real and fake video groups.

    Labels:
        0 = Real
        1 = Fake
    """

    real_frames = [
        frame
        for video in real_videos
        for frame in video
        if frame is not None
    ]

    fake_frames = [
        frame
        for video in fake_videos
        for frame in video
        if frame is not None
    ]

    if not real_frames:
        raise ValueError("No real frames found.")

    if not fake_frames:
        raise ValueError("No fake frames found.")

    real_frames = np.stack(real_frames).astype(np.uint8)
    fake_frames = np.stack(fake_frames).astype(np.uint8)

    images = np.concatenate(
        [real_frames, fake_frames],
        axis=0
    )

    real_labels = np.zeros(
        len(real_frames),
        dtype=np.uint8
    )

    fake_labels = np.ones(
        len(fake_frames),
        dtype=np.uint8
    )

    labels = np.concatenate(
        [real_labels, fake_labels],
        axis=0
    )

    return images, labels
# Training data
train_ff, train_ff_labels = combine_split(
    ff_real_train_f,
    ff_fake_train_f
)

# Validation data
val_ff, val_ff_labels = combine_split(
    ff_real_val_f,
    ff_fake_val_f
)

# Testing data
test_ff, test_ff_labels = combine_split(
    ff_real_test_f,
    ff_fake_test_f
)
print("\nTRAIN")
print("Images:", train_ff.shape)
print("Labels:", train_ff_labels.shape)
print("Real:", np.sum(train_ff_labels == 0))
print("Fake:", np.sum(train_ff_labels == 1))

print("\nVALIDATION")
print("Images:", val_ff.shape)
print("Labels:", val_ff_labels.shape)
print("Real:", np.sum(val_ff_labels == 0))
print("Fake:", np.sum(val_ff_labels == 1))

print("\nTEST")
print("Images:", test_ff.shape)
print("Labels:", test_ff_labels.shape)
print("Real:", np.sum(test_ff_labels == 0))
print("Fake:", np.sum(test_ff_labels == 1))

print("\nData types")
print("Train images:", train_ff.dtype)
print("Train labels:", train_ff_labels.dtype)

Reloaded main splits. Example shape: (160, 160, 3)
Real train videos: 517 | Fake train videos: 320

TRAIN
Images: (4595, 160, 160, 3)
Labels: (4595,)
Real: 2948
Fake: 1647

VALIDATION
Images: (948, 160, 160, 3)
Labels: (948,)
Real: 499
Fake: 449

TEST
Images: (2475, 160, 160, 3)
Labels: (2475,)
Real: 1444
Fake: 1031

Data types
Train images: uint8
Train labels: uint8


In [10]:
# ============================================================
# DATASETS
# ============================================================
train_dataset = DeepfakeDinoV2Dataset(images=train_ff,labels=train_ff_labels,processor=processor)
val_dataset = DeepfakeDinoV2Dataset(images=val_ff,labels=val_ff_labels,processor=processor)
test_dataset = DeepfakeDinoV2Dataset(images=test_ff,labels=test_ff_labels,processor=processor)
# ============================================================
# DATALOADERS
# ============================================================
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())


print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training samples: 4595
Validation samples: 948
Testing samples: 2475
Training batches: 288
Validation batches: 60
Testing batches: 155


In [ ]:
history = train_dinov2(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=EPOCHS
)


Epoch 01/10 | Train Loss: 0.0393 | Train Accuracy: 0.6578 | Val Loss: 0.0350 | Val Accuracy: 0.7277  
Epoch 02/10 | Train Loss: 0.0363 | Train Accuracy: 0.7232 | Val Loss: 0.0346 | Val Accuracy: 0.7285  
Epoch 03/10 | Train Loss: 0.0360 | Train Accuracy: 0.7264 | Val Loss: 0.0343 | Val Accuracy: 0.7325  
Epoch 04/10 | Train Loss: 0.0354 | Train Accuracy: 0.7267 | Val Loss: 0.0326 | Val Accuracy: 0.7428  
Epoch 05/10 | Train Loss: 0.0319 | Train Accuracy: 0.7515 | Val Loss: 0.0297 | Val Accuracy: 0.7588  
Epoch 06/10 | Train Loss: 0.0292 | Train Accuracy: 0.7783 | Val Loss: 0.0294 | Val Accuracy: 0.7707  
Epoch 07/10 | Train Loss: 0.0296 | Train Accuracy: 0.7804 | Val Loss: 0.0274 | Val Accuracy: 0.7874  
Epoch 08/10 | Train Loss: 0.0253 | Train Accuracy: 0.8198 | Val Loss: 0.0252 | Val Accuracy: 0.8065  
Epoch 09/10 | Train Loss: 0.0219 | Train Accuracy: 0.8482 | Val Loss: 0.0263 | Val Accuracy: 0.8248  
Epoch 10/10 | Train Loss: 0.0183 | Train Accuracy: 0.8837 | Val Loss: 0.0248 | Va

In [12]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [13]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [ ]:
test_results, confusion, report, predictions_df = (
    evaluate_dinov2(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        return_details=True
    )
)
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")
print(confusion)

print("\nConfusion Matrix Format:")
print("[[TN, FP],")
print(" [FN, TP]]")

print("\nClassification Report:")
print(report)



DINOV2-BASE TEST RESULTS
test_loss                     : 0.691900

Confusion Matrix:
accuracy                      : 0.8303

Confusion Matrix:
balanced_accuracy             : 0.8403

Confusion Matrix:
precision                     : 0.7454

Confusion Matrix:
recall_sensitivity            : 0.9001

Confusion Matrix:
specificity                   : 0.7805

Confusion Matrix:
f1_score                      : 0.8155

Confusion Matrix:
mcc                           : 0.6710

Confusion Matrix:
roc_auc                       : 0.8403

Confusion Matrix:
pr_auc                        : 0.8512

Confusion Matrix:
average_precision             : 0.8512

Confusion Matrix:
eer                           : 0.104478

Confusion Matrix:
eer_threshold                 : 0.519667

Confusion Matrix:
false_positive_rate           : 0.2195

Confusion Matrix:
false_negative_rate           : 0.0999

Confusion Matrix:
true_negatives                : 1127

Confusion Matrix:
false_positives               : 317

Confu

In [15]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 14.7%
Time Usage: 36.9 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [16]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 15.2%
Time Usage: 39.0 s
GPU Memory Used: 1349.5 MB
Power Consumption: 93W


#save the model

In [17]:
# ============================================================
# SAVE DINOV2-BASE CLASSIFIER
# ============================================================

import os
import torch


SAVE_DIR = os.path.join(
    r"D:\thesis\results",
    "dinov2_base_ff_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


WEIGHTS_PATH = os.path.join(
    SAVE_DIR,
    "dinov2_classifier_weights.pth"
)

CHECKPOINT_PATH = os.path.join(
    SAVE_DIR,
    "training_checkpoint.pt"
)


# ============================================================
# SAVE PROCESSOR
# ============================================================

processor.save_pretrained(
    SAVE_DIR
)


# ============================================================
# SAVE MODEL WEIGHTS ONLY
# ============================================================

torch.save(
    model.state_dict(),
    WEIGHTS_PATH
)


# ============================================================
# SAVE COMPLETE TRAINING CHECKPOINT
# ============================================================

completed_epochs = len(
    history.get(
        "train_loss",
        []
    )
)


checkpoint = {
    # Complete DINOv2 backbone and classifier-head weights
    "model_state_dict":
        model.state_dict(),

    # Optimizer state for resuming training
    "optimizer_state_dict":
        optimizer.state_dict(),

    # Training and validation history
    "history":
        history,

    # Model configuration
    "model_name":
        MODEL_NAME,

    "input_size":
        INPUT_SIZE,

    "num_classes":
        NUM_CLASSES,

    "dropout_rate":
        0.3,

    # Training configuration
    "completed_epochs":
        completed_epochs,

    "configured_epochs":
        EPOCHS,

    "batch_size":
        BATCH_SIZE,

    "learning_rate":
        LEARNING_RATE,

    "random_seed":
        RANDOM_SEED,

    # Label interpretation
    "label_mapping": {
        0: "real",
        1: "fake"
    },

    # Processor preprocessing information
    "image_mean":
        processor.image_mean,

    "image_std":
        processor.image_std,

    "processor_size":
        processor.size
}


torch.save(
    checkpoint,
    CHECKPOINT_PATH
)


# ============================================================
# VERIFY SAVED FILES
# ============================================================

print("DINOv2 model saved successfully.")
print("Save directory:", SAVE_DIR)
print("Weights:", WEIGHTS_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

print("\nSaved files:")

for filename in os.listdir(SAVE_DIR):

    file_path = os.path.join(
        SAVE_DIR,
        filename
    )

    if os.path.isfile(file_path):

        file_size_mb = (
            os.path.getsize(file_path)
            / (1024 ** 2)
        )

        print(
            f"{filename:40s}: "
            f"{file_size_mb:.2f} MB"
        )

DINOv2 model saved successfully.
Save directory: D:\thesis\results\dinov2_base_ff_160
Weights: D:\thesis\results\dinov2_base_ff_160\dinov2_classifier_weights.pth
Checkpoint: D:\thesis\results\dinov2_base_ff_160\training_checkpoint.pt

Saved files:
dinov2_classifier_weights.pth           : 330.37 MB
preprocessor_config.json                : 0.00 MB
training_checkpoint.pt                  : 991.11 MB


In [18]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_41224\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [19]:
# ============================================================
# PYTORCH MAXVIT WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        # MaxViT dataset returns:
        # images tensor, labels tensor
        images, labels = batch

        images = images.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # timm MaxViT returns logits directly
        logits = model(images)

        # Materialize part of the output
        _ = logits[-1, 0].item()


if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting DinoV2 resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous CUDA operations remain queued
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for images, labels in test_loader:

            images = images.to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            # timm MaxViT forward pass
            logits = model(images)

            last_logits = logits

            processed_images += images.size(0)


    # Wait for all CUDA inference operations
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    if last_logits is None:
        monitor.stop(
            elapsed_seconds=0.0,
            number_of_images=0
        )

        raise RuntimeError(
            "No images were processed during inference."
        )


    last_output_value = float(
        last_logits[-1, 0]
        .detach()
        .cpu()
        .item()
    )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number

    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits
    del images


# ============================================================
# DISPLAY INDIVIDUAL RUNS
# ============================================================

results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual DinoV2 profiling runs:"
)

display(results_df)

Device: cuda
Test images: 2475
Test batches: 155

Performing warm-up using 3 batches...
Warm-up completed.

Starting DinoV2 resource run 1/5
Run 1: 22.45 seconds | 9.0715 ms/image | 110.23 images/s

Starting DinoV2 resource run 2/5
Run 2: 21.85 seconds | 8.8300 ms/image | 113.25 images/s

Starting DinoV2 resource run 3/5
Run 3: 22.03 seconds | 8.8999 ms/image | 112.36 images/s

Starting DinoV2 resource run 4/5
Run 4: 22.34 seconds | 9.0247 ms/image | 110.81 images/s

Starting DinoV2 resource run 5/5
Run 5: 22.23 seconds | 8.9825 ms/image | 111.33 images/s

Individual DinoV2 profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,22.452059,9.071539,110.234880,2.643684,6.718750,5774.914062,5783.062438,5783.503906,8.148375,8.589844,...,118.085106,120.0,35.781915,100,58.785915,140.956,0.372138,1,2475,0.047349
1,21.854369,8.830048,113.249666,2.625494,5.909375,5783.593750,5783.815281,5788.453125,0.221531,4.859375,...,117.966102,120.0,37.288136,100,61.322593,142.662,0.381045,2,2475,0.047349
2,22.027217,8.899886,112.360993,2.678455,5.375000,5783.667969,5783.836458,5788.207031,0.168490,4.539062,...,118.000000,120.0,37.372222,100,59.112311,142.838,0.368344,3,2475,0.047349
3,22.336012,9.024651,110.807606,2.623774,6.234375,5783.699219,5784.227445,5788.507812,0.528226,4.808594,...,118.064516,120.0,36.548387,100,58.436801,142.868,0.365527,4,2475,0.047349
4,22.231568,8.982452,111.328180,2.646190,6.718750,5784.312500,5784.448801,5787.968750,0.136301,3.656250,...,118.074866,120.0,36.550802,100,58.351652,144.401,0.364482,5,2475,0.047349


In [20]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("DinoV2 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


DinoV2 RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,22.180245,0.239996,21.882250,22.478240
1,latency_ms_per_image,8.961715,0.096968,8.841313,9.082117
2,throughput_images_per_s,111.596265,1.210967,110.092651,113.099879
3,average_cpu_percent,2.643519,0.022037,2.616157,2.670881
4,peak_cpu_percent,6.191250,0.570984,5.482280,6.900220
5,average_ram_mb,5783.878084,0.528849,5783.221431,5784.534738
6,peak_ram_mb,5787.328125,2.148535,5784.660366,5789.995884
7,average_incremental_ram_mb,1.840584,3.529601,-2.541994,6.223163
8,peak_incremental_ram_mb,5.290625,1.906515,2.923374,7.657876
9,average_gpu_memory_mb,3679.788118,0.052188,3679.723318,3679.852919



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 8.962 ± 0.097 (95% CI: 8.841–9.082)
peak_ram_mb: 5787.328 ± 2.149 (95% CI: 5784.660–5789.996)
peak_gpu_memory_mb: 3681.750 ± 0.000 (95% CI: 3681.750–3681.750)
average_gpu_utilization_percent: 36.708 ± 0.649 (95% CI: 35.902–37.514)
average_gpu_power_w: 59.202 ± 1.223 (95% CI: 57.683–60.721)


#gernalization

In [22]:
#wild deepfake on ff
print("\nTest results of FF++ on wild deepfake dataset DinoV2:")
test_dataset = DeepfakeDinoV2Dataset(images=test_images,labels=test_labels,processor=processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, report, predictions_df = (evaluate_dinov2(model=model,data_loader=test_loader,criterion=criterion,device=device,return_details=True ))
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")


Test results of FF++ on wild deepfake dataset SwinV2:

DINOV2-BASE TEST RESULTS
test_loss                     : 0.670289

Confusion Matrix:
accuracy                      : 0.586167

Confusion Matrix:
balanced_accuracy             : 0.444037

Confusion Matrix:
precision                     : 0.722251

Confusion Matrix:
recall_sensitivity            : 0.728296

Confusion Matrix:
specificity                   : 0.159778

Confusion Matrix:
f1_score                      : 0.725261

Confusion Matrix:
mcc                           : -0.112887

Confusion Matrix:
roc_auc                       : 0.395235

Confusion Matrix:
pr_auc                        : 0.680633

Confusion Matrix:
average_precision             : 0.680719

Confusion Matrix:
eer                           : 0.564370

Confusion Matrix:
eer_threshold                 : 0.622470

Confusion Matrix:
false_positive_rate           : 0.840222

Confusion Matrix:
false_negative_rate           : 0.271704

Confusion Matrix:
true_negatives    

In [24]:
#celeb on ff
print("\nTest results of FF++ on Celeb-df(v2) dataset (DinoV2):")
test_dataset = DeepfakeDinoV2Dataset(images=test_celeb,labels=test_labels,processor=processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, report, predictions_df = (evaluate_dinov2(model=model,data_loader=test_loader,criterion=criterion,device=device,return_details=True ))
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")


Test results of FF++ on Celeb-df(v2) dataset (DinoV2):

DINOV2-BASE TEST RESULTS
test_loss                     : 0.536759

Confusion Matrix:
accuracy                      : 0.824358

Confusion Matrix:
balanced_accuracy             : 0.523956

Confusion Matrix:
precision                     : 0.908868

Confusion Matrix:
recall_sensitivity            : 0.895547

Confusion Matrix:
specificity                   : 0.152364

Confusion Matrix:
f1_score                      : 0.902159

Confusion Matrix:
mcc                           : 0.045238

Confusion Matrix:
roc_auc                       : 0.518225

Confusion Matrix:
pr_auc                        : 0.901434

Confusion Matrix:
average_precision             : 0.901586

Confusion Matrix:
eer                           : 0.488557

Confusion Matrix:
eer_threshold                 : 0.633317

Confusion Matrix:
false_positive_rate           : 0.847636

Confusion Matrix:
false_negative_rate           : 0.104453

Confusion Matrix:
true_negatives    

In [26]:
#DFC on ff
print("\nTest results of FF++ on DFC dataset (Dinov2):")
test_dataset = DeepfakeDinoV2Dataset(images=test_hog,labels=test_labels,processor=processor)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results, confusion, report, predictions_df = (evaluate_dinov2(model=model,data_loader=test_loader,criterion=criterion,device=device,return_details=True ))
print("\nDINOV2-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )

    else:
        print(
            f"{metric:30s}: {value}"
        )
    print("\nConfusion Matrix:")


Test results of FF++ on DFC dataset (Dinov2):

DINOV2-BASE TEST RESULTS
test_loss                     : 0.700606

Confusion Matrix:
accuracy                      : 0.527667

Confusion Matrix:
balanced_accuracy             : 0.527667

Confusion Matrix:
precision                     : 0.518338

Confusion Matrix:
recall_sensitivity            : 0.782000

Confusion Matrix:
specificity                   : 0.273333

Confusion Matrix:
f1_score                      : 0.623439

Confusion Matrix:
mcc                           : 0.064269

Confusion Matrix:
roc_auc                       : 0.521053

Confusion Matrix:
pr_auc                        : 0.497551

Confusion Matrix:
average_precision             : 0.498267

Confusion Matrix:
eer                           : 0.476000

Confusion Matrix:
eer_threshold                 : 0.544583

Confusion Matrix:
false_positive_rate           : 0.726667

Confusion Matrix:
false_negative_rate           : 0.218000

Confusion Matrix:
true_negatives             